In [369]:
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.tools import tool
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage, ChatMessage
import json

load_dotenv()

API_KEY = "15b1cf57a44e1d91063cc56c"

In [351]:
@tool
def get_conversion_rate(base_currency: str, target_currency: str) -> float:
    """This function fetches the currency conversion factor between a given base currency and a target currency!!!"""
    result = requests.get(f"https://v6.exchangerate-api.com/v6/{API_KEY}/pair/{base_currency}/{target_currency}").json()
    conversion_rate = result['conversion_rate']

    return conversion_rate

@tool
def convert(base_currency_amount: float, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """Given a currency conversion rate this function calculates the target currency value from a given base currency value!!!"""

    return base_currency_amount * conversion_rate

In [352]:
print(get_conversion_rate.invoke({"base_currency":'USD', "target_currency":"INR"}))

93.3055


In [353]:
llm = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
)

model = ChatHuggingFace(llm=llm)

final_model = model.bind_tools([convert, get_conversion_rate])

In [354]:
messages = [HumanMessage("I want you to do the following tasks simulatneously: firstly, fetch the conversion rate from USD to INR and then secondly using the fetched conversion rate, convert 13 USD dollars to INR rupees...")]

In [355]:
messages

[HumanMessage(content='I want you to do the following tasks simulatneously: firstly, fetch the conversion rate from USD to INR and then secondly using the fetched conversion rate, convert 13 USD dollars to INR rupees...', additional_kwargs={}, response_metadata={})]

In [356]:
ai_message = final_model.invoke(messages)

In [357]:
ai_message

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_rate', 'description': None}, 'id': 'call_745fwhmstfcg0034a17zntwj', 'type': 'function'}, {'function': {'arguments': '{"base_currency_amount":13}', 'name': 'convert', 'description': None}, 'id': 'call_tcuyu9mvhv4tmgqlyee5cjay', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 367, 'total_tokens': 419}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d68e8-8ed3-7ec1-9efa-ad6c74ea28ed-0', tool_calls=[{'name': 'get_conversion_rate', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_745fwhmstfcg0034a17zntwj', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_amount': 13}, 'id': 'call_tcuyu9mvhv4tmgqlyee5cjay', 'type': 'tool_call'}], invalid_tool_calls=[], usage_m

In [361]:
messages.append(ai_message)

In [362]:
ai_tool_call = ai_message.tool_calls

In [363]:
ai_tool_call

[{'name': 'get_conversion_rate',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_745fwhmstfcg0034a17zntwj',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_amount': 13},
  'id': 'call_tcuyu9mvhv4tmgqlyee5cjay',
  'type': 'tool_call'}]

In [373]:
float(get_conversion_rate.invoke(ai_tool_call[0]).content)

93.3055

In [365]:
messages

[HumanMessage(content='I want you to do the following tasks simulatneously: firstly, fetch the conversion rate from USD to INR and then secondly using the fetched conversion rate, convert 13 USD dollars to INR rupees...', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_rate', 'description': None}, 'id': 'call_745fwhmstfcg0034a17zntwj', 'type': 'function'}, {'function': {'arguments': '{"base_currency_amount":13}', 'name': 'convert', 'description': None}, 'id': 'call_tcuyu9mvhv4tmgqlyee5cjay', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 367, 'total_tokens': 419}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d68e8-8ed3-7ec1-9efa-ad6c74ea28ed-0', tool_calls=[{'name': 'get_conversion_rate', 'ar

In [374]:
for tool_call in ai_tool_call:
    if tool_call['name'] == 'get_conversion_rate':
        tool_msg1 = get_conversion_rate.invoke(tool_call)
        conversion_rate = float(tool_msg1.content)
        messages.append(tool_msg1)

    if tool_call['name'] == 'convert':
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_msg2 = convert.invoke(tool_call)
        messages.append(tool_msg2)

In [375]:
result = final_model.invoke(messages)

In [377]:
result.content

'The current conversion rate from USD to INR is approximately 93.31 (rounded to two decimal places).\n\nUsing this conversion rate, 13 USD dollars is approximately 1213.00 INR rupees (rounded to two decimal places).'

In [378]:
messages

[HumanMessage(content='I want you to do the following tasks simulatneously: firstly, fetch the conversion rate from USD to INR and then secondly using the fetched conversion rate, convert 13 USD dollars to INR rupees...', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_rate', 'description': None}, 'id': 'call_745fwhmstfcg0034a17zntwj', 'type': 'function'}, {'function': {'arguments': '{"base_currency_amount":13}', 'name': 'convert', 'description': None}, 'id': 'call_tcuyu9mvhv4tmgqlyee5cjay', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 367, 'total_tokens': 419}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d68e8-8ed3-7ec1-9efa-ad6c74ea28ed-0', tool_calls=[{'name': 'get_conversion_rate', 'ar